# CS F425 Phase 1 Live Demo

Loads the Phase 1 Phi-2 QLoRA adapter and runs a user prompt against the sales-data agent.

In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install --no-cache-dir --upgrade "bitsandbytes>=0.45.5" "accelerate" "transformers" "peft" "pandas"


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
import gc
import glob as _glob
import json
import os
import re
import sys
from numbers import Integral, Real

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("This live demo expects a GPU runtime. In Colab, switch to a T4 GPU runtime.")

DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
MODEL_NAME = "microsoft/phi-2"
ADAPTER_DIR = f"{DRIVE_DIR}/phi2-agent-adapter"
CKPT_DIR = f"{DRIVE_DIR}/phi2-agent-qlora"
MAX_SEQ_LENGTH = 384

sys.path.insert(0, DRIVE_DIR)
from tool_executor import ToolExecutor

df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")


def latest_path(paths, key_fn):
    if not paths:
        return None
    return sorted(paths, key=key_fn)[-1]


def resolve_phase1_load_path():
    epoch_ckpts = _glob.glob(f"{CKPT_DIR}/checkpoint-*")
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR}/timed_ckpt_step_*")
    final_adapter_exists = os.path.isfile(f"{ADAPTER_DIR}/adapter_config.json")

    print("Phase 1 model:", MODEL_NAME)
    print("Final adapter dir:", ADAPTER_DIR)
    print("Trainer checkpoints dir:", CKPT_DIR)
    print("epoch checkpoints count:", len(epoch_ckpts))
    print("timed checkpoints count:", len(timed_ckpts))
    print("final adapter exists:", final_adapter_exists)

    if final_adapter_exists:
        return ADAPTER_DIR
    if epoch_ckpts:
        return latest_path(epoch_ckpts, lambda p: int(p.rsplit("-", 1)[-1]))
    if timed_ckpts:
        return latest_path(timed_ckpts, lambda p: int(p.rsplit("_", 1)[-1]))
    raise ValueError("No saved Phase 1 adapter or checkpoint found in Drive.")


LOAD_PATH = resolve_phase1_load_path()
print("Loading Phase 1 weights from:", LOAD_PATH)

try:
    del model
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, LOAD_PATH)
model.eval()
model.config.use_cache = True

print("Phase 1 Phi-2 model loaded and ready.")


In [ ]:
SCHEMA = """Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)"""

PROMPT_TEMPLATE = """### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an \"actions\" list and an \"answer\" field. No other text.

### Schema
{schema}

### Question
{question}

### Answer
"""


def make_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)


def extract_json_object(raw: str):
    raw = str(raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    return {"raw_output": raw}


def generate_phase1_json(question: str, max_new_tokens: int = 220):
    prompt = make_prompt(question)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return extract_json_object(raw), raw


In [ ]:
def _parse_numeric(val_str):
    return float(val_str) if "." in str(val_str) else int(val_str)


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def _split_args(arg_text):
    args = []
    cur = ""
    quote = None
    depth = 0

    for ch in arg_text:
        if quote:
            cur += ch
            if ch == quote:
                quote = None
        else:
            if ch in ["'", '"']:
                quote = ch
                cur += ch
            elif ch in "([{" :
                depth += 1
                cur += ch
            elif ch in ")]}":
                depth -= 1
                cur += ch
            elif ch == "," and depth == 0:
                if cur.strip():
                    args.append(cur.strip())
                cur = ""
            else:
                cur += ch

    if cur.strip():
        args.append(cur.strip())
    return args


def _parse_arg_value(value):
    value = value.strip()

    if len(value) >= 2 and value[0] == value[-1] and value[0] in ["'", '"']:
        return value[1:-1]
    if re.fullmatch(r"-?\d+", value):
        return int(value)
    if re.fullmatch(r"-?\d+\.\d+", value):
        return float(value)
    if value.lower() == "true":
        return True
    if value.lower() == "false":
        return False
    return value


def parse_tool_invocation(invocation):
    invocation = str(invocation).strip()
    invocation = invocation.replace("\u2019", "'").replace("\u2018", "'")
    invocation = invocation.replace("\u201c", '"').replace("\u201d", '"')
    match = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s*\((.*)\)\s*$", invocation, re.DOTALL)

    if not match:
        return None

    name = match.group(1).strip()
    arg_text = match.group(2).strip()
    parsed_args = {}

    if arg_text:
        for part in _split_args(arg_text):
            if "=" not in part:
                continue
            key, value = part.split("=", 1)
            parsed_args[key.strip()] = _parse_arg_value(value)

    return name, parsed_args


def parse_agent_action(action_str):
    parsed = parse_tool_invocation(action_str)
    if parsed is None:
        return None

    name, args = parsed

    if name == "filter_data":
        return {"tool": "filter", "args": {"column": str(args["column"]), "op": "==", "value": args["value"]}}
    if name == "group_by":
        return {"tool": "groupby", "args": {"column": str(args["column"])}}
    if name == "aggregate_sum":
        return {"tool": "aggregate", "args": {"column": str(args["column"]), "agg": "sum"}}
    if name == "aggregate_mean":
        return {"tool": "aggregate", "args": {"column": str(args["column"]), "agg": "mean"}}
    if name == "aggregate_count":
        return {"tool": "aggregate", "args": {"column": str(args["column"]), "agg": "count"}}
    if name == "sort_by":
        return {"tool": "sort", "args": {"column": str(args["column"]), "ascending": str(args.get("order", "desc")).lower() != "desc"}}
    if name == "top_k":
        return {"tool": "topk", "args": {"k": int(args["k"])}}

    return None


def execute_action_sequence(actions, source_df):
    parsed_actions = []
    for action in actions:
        parsed_action = parse_agent_action(action)
        if parsed_action is None:
            raise ValueError(f"Could not parse action: {action}")
        parsed_actions.append(parsed_action)
    return ToolExecutor(source_df.copy()).execute(parsed_actions)


def result_to_python(actions, result_df):
    if result_df is None:
        return None
    if hasattr(result_df, "empty") and result_df.empty:
        return None
    if hasattr(result_df, "groups") and not isinstance(result_df, pd.DataFrame):
        return None

    action_names = [action.split("(", 1)[0] for action in actions]
    has_groupby = any(name == "group_by" for name in action_names)
    has_sort_or_topk = any(name in {"sort_by", "top_k"} for name in action_names)

    if getattr(result_df, "shape", None) == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])
    if isinstance(result_df, pd.DataFrame):
        if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
            key_col, value_col = result_df.columns
            return {str(row[key_col]): clean_scalar(row[value_col]) for _, row in result_df.iterrows()}
        return [{str(key): clean_scalar(value) for key, value in row.items()} for row in result_df.to_dict(orient="records")]

    return clean_scalar(result_df)


def compute_answer(actions, source_df):
    if not actions:
        return None
    try:
        result_df = execute_action_sequence(actions, source_df)
        return result_to_python(actions, result_df)
    except Exception as exc:
        return {"executor_error": str(exc)}


In [ ]:
def run_phase1_query(question: str, execute_actions: bool = True, show_raw: bool = False):
    model_json, raw_output = generate_phase1_json(question)

    actions = []
    model_answer = None
    if isinstance(model_json, dict):
        actions = model_json.get("actions", []) or []
        model_answer = model_json.get("answer")

    executor_answer = compute_answer(actions, df) if execute_actions and actions else None
    final_answer = executor_answer if executor_answer is not None else model_answer

    final_output = {
        "actions": actions,
        "answer": final_answer,
    }

    if show_raw:
        print("Raw model output:")
        print(raw_output)
        print()

    print(json.dumps(final_output, indent=2, ensure_ascii=False))
    return final_output


query = input("Enter query: ")
_ = run_phase1_query(query, execute_actions=True, show_raw=False)
